# Minimal Top-K Overlap (Log Only)

给定多个 `prompt -> seed->score` JSON，计算 top-k 重叠并直接打印结果。

当前默认规则：
- `TOP_K=2`
- `AT_LEAST_ONE_OVERLAP=True`（两组 top2 只要有 1 个 seed 相同就算重叠）
- 不保存任何文件，只输出 log。

In [5]:
from pathlib import Path
from itertools import combinations
import json

# ===== Config =====
TOP_K = 2
AT_LEAST_ONE_OVERLAP = True  # True: |A∩B|>=1; False: exact top-k set equal

JSON_FILES = {
    "t1": "/Users/loskiwu/Documents/Loski/Work/Intern/VR/work1_LRM/code/VBench/resources/lrm_result/scores_t_1.json",
    "t5": "/Users/loskiwu/Documents/Loski/Work/Intern/VR/work1_LRM/code/VBench/resources/lrm_result/scores_t_5.json",
    "t10": "/Users/loskiwu/Documents/Loski/Work/Intern/VR/work1_LRM/code/VBench/resources/lrm_result/scores_t_10.json",
    "t50": "/Users/loskiwu/Documents/Loski/Work/Intern/VR/work1_LRM/code/VBench/resources/lrm_result/scores_t_50.json",
}

# JSON_FILES = {
#     "ur": "/Users/loskiwu/Documents/Loski/Work/Intern/VR/work1_LRM/code/VBench/resources/previous_rm_result/clean/ur.json",
#     "vr": "/Users/loskiwu/Documents/Loski/Work/Intern/VR/work1_LRM/code/VBench/resources/previous_rm_result/clean/vr.json",
#     "vs2": "/Users/loskiwu/Documents/Loski/Work/Intern/VR/work1_LRM/code/VBench/resources/previous_rm_result/clean/vs2.json",
# }


def load_json(path: str):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def topk(seed_scores: dict, k: int):
    items = sorted(seed_scores.items(), key=lambda kv: (-float(kv[1]), int(kv[0])))
    return [s for s, _ in items[:k]]


scores = {name: load_json(path) for name, path in JSON_FILES.items()}
names = list(scores.keys())
prompts = sorted(next(iter(scores.values())).keys())

# Build top-k map
kmap = {
    name: {p: topk(scores[name][p], TOP_K) for p in prompts}
    for name in names
}

print(f"prompts={len(prompts)}, models={names}, top_k={TOP_K}")

# Pairwise overlap logs
for a, b in combinations(names, 2):
    overlap = 0
    no_overlap = 0
    for p in prompts:
        inter = len(set(kmap[a][p]) & set(kmap[b][p]))
        ok = inter >= 1 if AT_LEAST_ONE_OVERLAP else inter == TOP_K
        if ok:
            overlap += 1
        else:
            no_overlap += 1

    ratio = overlap / len(prompts)
    print(f"{a}-{b}: overlap={overlap}/{len(prompts)} ({ratio:.4f}), no_overlap={no_overlap}")

# 3-way overlap log: intersection among all sets non-empty
if len(names) >= 3:
    three = 0
    for p in prompts:
        inter = set(kmap[names[0]][p])
        for n in names[1:]:
            inter &= set(kmap[n][p])
        if len(inter) >= 1:
            three += 1
    print(f"all-model shared-at-least-one={three}/{len(prompts)} ({three/len(prompts):.4f})")

prompts=946, models=['t1', 't5', 't10', 't50'], top_k=2
t1-t5: overlap=645/946 (0.6818), no_overlap=301
t1-t10: overlap=664/946 (0.7019), no_overlap=282
t1-t50: overlap=690/946 (0.7294), no_overlap=256
t5-t10: overlap=669/946 (0.7072), no_overlap=277
t5-t50: overlap=664/946 (0.7019), no_overlap=282
t10-t50: overlap=670/946 (0.7082), no_overlap=276
all-model shared-at-least-one=126/946 (0.1332)
